# Introduction to NLP Fundamentals in TensorFlow

NLP has the goal of deriving information out of natural language (could be sequences of text or speech).

Another common term for NLP problems is sequence to sequence problems (seq2seq).

In [1]:
# ============================================================================
# ⚡ GEEKOM A9 MAX - Setup Rápido para Estudos
# Cole no início de qualquer notebook e execute primeiro
# ============================================================================

import os, warnings
os.environ.update({
    'TF_CPP_MIN_LOG_LEVEL': '2',           # Menos logs
    'TF_ENABLE_ONEDNN_OPTS': '1',          # oneDNN (2-3x mais rápido)
    'OMP_NUM_THREADS': '32',               # 32 threads
    'MKL_NUM_THREADS': '32',
    'TF_NUM_INTEROP_THREADS': '4',
    'TF_NUM_INTRAOP_THREADS': '32',
    'TF_XLA_FLAGS': '--tf_xla_auto_jit=2', # XLA JIT
    'CUDA_VISIBLE_DEVICES': '-1',          # CPU only
})
warnings.filterwarnings('ignore')

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Config TensorFlow
try: 
    tf.config.optimizer.set_jit(True)
    tf.config.threading.set_inter_op_parallelism_threads(4)
    tf.config.threading.set_intra_op_parallelism_threads(32)
except: pass
tf.config.set_visible_devices([], 'GPU')

# Gráficos
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = [14, 6]

print(f"✅ TensorFlow {tf.__version__} | Ryzen 9 (32t) | 96GB RAM | oneDNN ativo")
print("="*80)


✅ TensorFlow 2.15.0 | Ryzen 9 (32t) | 96GB RAM | oneDNN ativo


In [2]:
# Import series of helper functions for the notebook
from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

## Get a text dataset

The dataset we're going to be using is Kaggle's introduction to NLP dataset (text samples of Tweets labelled as disaster or not disaster).

In [3]:
# import zipfile
# import urllib.request

# # Baixar o arquivo ZIP
# url = "https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip"
# zip_path = "nlp_getting_started.zip"
# urllib.request.urlretrieve(url, zip_path)

# # # Descompactar o arquivo ZIP
# with zipfile.ZipFile(zip_path, "r") as zip_ref:
#     zip_ref.extractall()

## Visualization of the data (text)

Once you've acquired a new dataset to work with, what should you do first?

Explore it? Inspect it? Verify it? Become one with it?

All correct.

Remember the motto: visualize, visualize, visualize.

Right now, our text data samples are in the form of '.csv' files. For an easy way to make them visual, let's turn them into pandas DataFrame's.

📖 Reading: You might come across text datasets in many different formats. Aside from CSV files (what we're working with), you'll probably encounter '.txt' files and '.json' files too. In this section, we'll learn how to read these types of files as well. For working with these type of files, I'd recommend reading the two following articles by RealPython:

* [How to Read and Write Files in Python](https://realpython.com/read-write-files-python/)
* [How to Read and Write JSON Files in Python](https://realpython.com/python-json/)

In [4]:
import pandas as pd
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
print(train_df.head())

   id keyword location                                               text  \
0   1     NaN      NaN  Our Deeds are the Reason of this #earthquake M...   
1   4     NaN      NaN             Forest fire near La Ronge Sask. Canada   
2   5     NaN      NaN  All residents asked to 'shelter in place' are ...   
3   6     NaN      NaN  13,000 people receive #wildfires evacuation or...   
4   7     NaN      NaN  Just got sent this photo from Ruby #Alaska as ...   

   target  
0       1  
1       1  
2       1  
3       1  
4       1  


In [5]:
# Shuffle training dataframe
train_df_shuffled = train_df.sample(frac=1, random_state=42) # random_state is set to 42 for reproducibility
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [6]:
# What does the test dataframe look like?
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [7]:
# How many examples of each class are in the training set?
train_df.target.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

In [8]:
# How many total samples?
len(train_df), len(test_df)

(7613, 3263)

In [9]:
# Let's visualize some random training examples
import random
random_index = random.randint(0, len(train_df)-5)
for row in train_df_shuffled[["text", "target"]][random_index:random_index+5].itertuples():
    _, text, target = row
    print(f"Target: {target}", "(real threat)" if target > 0 else "(not a threat)")
    print(f"Text:\n{text}\n")
    print("---\n")

Target: 0 (not a threat)
Text:
@Stephen_Georg Hey Stephen Remember that time you drowned all the yellows

Read: http://t.co/0sa6Xx1oQ7

---

Target: 0 (not a threat)
Text:
This is interesting!--Why did God order obliteration of ancient Canaanites? http://t.co/XqMJHIOZxG via @worldnetdaily

---

Target: 0 (not a threat)
Text:
@SergioPiaggio 'IÛªd worked so hard to get to that level that I wasnÛªt going to let the injury define me. I was going to define it.Û Cool

---

Target: 1 (real threat)
Text:
Obama Declares Disaster for Typhoon-Devastated Saipan: Obama signs disaster declaration for Northern Marians a... http://t.co/JCszCJiHlH

---

Target: 0 (not a threat)
Text:
What a feat! Watch the #BTS of @kallemattson's incredible music video for #Avalanche: https://t.co/3W6seA9tuv ????

---



### Split data into training and valiation sets

In [10]:
# split the data into training and testing sets
from sklearn.model_selection import train_test_split

In [11]:
# Use train_test_split to split the data into training and testing sets.
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1,  # 10% of the data
                                                                            random_state=42) # random seed for reproducibility

In [12]:
# Check the length of the text.
len(train_sentences), len(train_labels), len(val_sentences), len(val_labels)

(6851, 6851, 762, 762)

In [13]:
# Check  the first 10 samples
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

## Converting text into numbers

When dealing with a text problem, one of the first things you'll have to do before you can build a model is to convert your text to numbers.
There are a few ways to do this, namely:
- Tokenization – direct mapping of a token (a token could be a word or a character) to a number
- Embedding – create a matrix of feature vectors for each token (the size of the feature vector can be defined and this embedding can be learned)

In [14]:
# convert text to numbers.
from tensorflow.keras.layers.experimental.preprocessing import TextVectorization

text_vectorizer = TextVectorization(max_tokens=10000, # how many words in the vocabulary (automatically add <OOV>)
                                    standardize="lower_and_strip_punctuation", # convert text to lowercase and remove punctuation
                                    split="whitespace", # split text into words or characters
                                    ngrams=None, # create groups of words or characters
                                    output_mode="int", # how to convert tokens to numbers
                                    output_sequence_length=None, # how long is the output sequence
                                    pad_to_max_tokens=True) # pad the sequence to the max length 

In [15]:
len(train_sentences[0].split()) # how many words in the first tweet

7

In [16]:
# Finde the average numbers of tokens (words) in the training tweets

round(sum([len(tweet.split()) for tweet in train_sentences])/len(train_sentences))

15

In [17]:
# Setup text Vectorization variables
max_vocab_length = 10000 # how many unique words in the vocabulary
max_length = 15 # max length of a text to consider

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [18]:
# Fit the text vectorizer on the training tweets
text_vectorizer.adapt(train_sentences)


In [19]:
# Create a sample sentence and tokenize it
sample_sentence = " There's a flood in my street!"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[264,   3, 232,   4,  13, 698,   0,   0,   0,   0,   0,   0,   0,
          0,   0]], dtype=int64)>

In [20]:
# Choose a random sentence from the training dataset and tokenize it
random_sentence = random.choice(train_sentences)   
print(f"Original tweet: \n {random_sentence}\
      \n\nVectorized Version:")
text_vectorizer([random_sentence])

Original tweet: 
 Flood Advisory in effect for Shelby County in AL until 9 PM #alwx http://t.co/gTqMGsgcsB      

Vectorized Version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[ 232, 2200,    4,  780,   10, 3426,  583,    4, 1906,  293,  491,
         176,    1,    1,    0]], dtype=int64)>

In [21]:
# get the unique words in the vocabulary
words_in_vocab = text_vectorizer.get_vocabulary() # get all the unique words in the vocabulary
top_5_words = words_in_vocab[:5] # get the top 5 words in the vocabulary
bottom_5_words = words_in_vocab[-5:] # get the bottom 5 words in the vocabulary
print(f"Numbers of words in vocab: {len(words_in_vocab)}")
print(f"5 most commom words: {top_5_words}")
print(f"5 least commom words: {bottom_5_words}")

Numbers of words in vocab: 10000
5 most commom words: ['', '[UNK]', 'the', 'a', 'in']
5 least commom words: ['pages', 'paeds', 'pads', 'padres', 'paddytomlinson1']


### Creating and Embedding using Embedding Layer

Creating an Embedding using an Embedding Layer
We've got a way to map our text to numbers. How about we go a step further and turn those numbers into an embedding?

The powerful thing about an embedding is it can be learned during training. This means rather than just being static (e.g. 1 = I, 2 = love, 3 = TensorFlow), a word's numeric representation can be improved as a model goes through data samples.

We can see what an embedding of a word looks like by using the [tf.keras.layers.Embedding laye](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding)r.

The main parameters we're concerned about here are:

* `input_dim` - The size of the vocabulary (e.g. len(text_vectorizer.get_vocabulary()).
* `output_dim` - The size of the output embedding vector, for example, a value of 100 outputs a feature vector of size 100 for each word.
* `embeddings_initializer` - How to initialize the embeddings matrix, default is "uniform" which randomly initalizes embedding matrix with uniform distribution. This can be changed for using pre-learned embeddings.
* `input_length` - Length of sequences being passed to embedding layer.
  
Knowing these, let's make an embedding layer.

In [22]:
from tensorflow.keras import layers

embedding = layers.Embedding(input_dim=max_vocab_length,
                            output_dim=128,
                            input_length=max_length,
                            )

In [23]:
# get a random sentence from the training set
random_sentence = random.choice(train_sentences)
print(f"Original tweet: \n {random_sentence}\
            \n\nEmbedded Version:")

# Embed the radon sentence (turn it into a dense vector of fixed sizes)
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed 

Original tweet: 
 #Saudi Arabia: #Abha: Fatalities reported following suicide bombing at mosque; avoid area http://t.co/1xW0Z8ZeqW            

Embedded Version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[-0.03331681,  0.01542809, -0.01488029, ...,  0.01881877,
         -0.04829221,  0.02407196],
        [ 0.01153781, -0.03233669,  0.02360667, ..., -0.04097359,
          0.02080715,  0.02400091],
        [ 0.03247278,  0.04027596,  0.03857771, ..., -0.00917685,
          0.03954077, -0.0428756 ],
        ...,
        [ 0.03247278,  0.04027596,  0.03857771, ..., -0.00917685,
          0.03954077, -0.0428756 ],
        [-0.00039382, -0.01750038, -0.0471017 , ..., -0.04472769,
         -0.01696729, -0.03934587],
        [-0.00039382, -0.01750038, -0.0471017 , ..., -0.04472769,
         -0.01696729, -0.03934587]]], dtype=float32)>

In [24]:
# Check out a single token's embedding
sample_embed[0][0], sample_embed[0][0].shape, random_sentence

(<tf.Tensor: shape=(128,), dtype=float32, numpy=
 array([-0.03331681,  0.01542809, -0.01488029,  0.02584905,  0.00543106,
         0.02407558,  0.03455068,  0.00113524, -0.01488752,  0.04575429,
        -0.01570852,  0.01490552, -0.00055669, -0.01594492, -0.02660613,
        -0.0497523 , -0.02199038,  0.0041078 , -0.01412736,  0.04345608,
         0.02490684,  0.0355312 ,  0.00253936, -0.04911355, -0.02102089,
        -0.01787686,  0.04255544,  0.0158394 ,  0.01829654, -0.00483675,
        -0.04883896,  0.03180626, -0.01410266,  0.01235765,  0.0354825 ,
         0.02691723, -0.00071125, -0.04847655,  0.00317334,  0.04564936,
        -0.01681061,  0.01921221, -0.00355345, -0.03708124,  0.0384461 ,
        -0.0216152 ,  0.04352361, -0.02580328,  0.00938385, -0.04488627,
         0.02885716, -0.02875379,  0.03696401,  0.01042304,  0.01364363,
         0.01374978, -0.04831014, -0.04867419,  0.01198936,  0.03690174,
         0.03844966,  0.02878397,  0.02795318, -0.00151662, -0.0176443 ,
  

### Modelling a text dataset (running a series of experiments)
Now we've got a way to turn our text sequences into numbers, it's time to start building a series of modelling experiments.
We'll start with a baseline and move on from there.
- Model 0: Naive Bayes (baseline), this is from Sklearn ML map: https://scikit-learn.org/stable/tutorial/machine_learning_map/index.html (scikit-learn.org in Bing)
- Model 1: Feed-forward neural network (dense model)
- Model 2: LSTM model (RNN)
- Model 3: GRU model (RNN)
- Model 4: Bidirectional-LSTM model (RNN)
- Model 5: 1D Convolutional Neural Network (CNN)
- Model 6: TensorFlow Hub Pretrained Feature Extractor (using transfer learning for NLP)
- Model 7: Same as model 6 with 10% of training data


How are we going to approach all of these?
Use the standard steps in modelling with TensorFlow:
- Create a model
- Build a model
- Fit a model
- Evaluate our model


### Model 0: Getting a baseline

As with all machine learning modelling experiments, it's important to create a baseline model so you've got a benchmark for future experiments to build upon.
To create our baseline, we'll use Sklearn's Multinomial Naive Bayes using the TF‑IDF formula to convert our words to numbers.
> 🔑 Note: It's common practice to use non‑DL algorithms as a baseline because of their speed and then later use DL to see if you can improve upon them.



In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
# Create the pipeline

model_0 = Pipeline([
    ('tfidf', TfidfVectorizer()), # convert text to numbers
    ('clf', MultinomialNB()) # model the text
])

# Fit the pipeline to the training data
model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [26]:
# Evaluate our baseline model
baseline_score = model_0.score(train_sentences, train_labels)
print(f"Our baseline model achieves an accuracy of: {baseline_score:.3f}%")

Our baseline model achieves an accuracy of: 0.887%


In [27]:
# Make predictions
baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
      dtype=int64)

### Creating an evaluation function for our model experiments
We could evaluate these as they are but since we're going to be evaluating several models in the same way going forward, let's create a helper function which takes an array of predictions and ground truth labels and computes the following:

* Accuracy
* Precision
* Recall
* F1-score
> 🔑 Note: Since we're dealing with a classification problem, the above metrics are the most appropriate. If we were working with a regression problem, other metrics such as MAE (mean absolute error) would be a better choice. We can find more at: https://scikit-learn.org/0.16/modules/model_evaluation.html

In [28]:
# Function to evaluate: accuracy, precision, recall, f1-score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall and f1 score of a binary classification model.

  Args:
  -----
  y_true = true labels in the form of a 1D array
  y_pred = predicted labels in the form of a 1D array

  Returns a dictionary of accuracy, precision, recall, f1-score.
  """
  # Calculate model accuracy
  model_accuracy = accuracy_score(y_true, y_pred) * 100
  # Calculate model precision, recall and f1 score using "weighted" average
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  model_results = {"accuracy": model_accuracy,
                  "precision": model_precision,
                  "recall": model_recall,
                  "f1": model_f1}
  return model_results

In [29]:
# get baseline results
baseline_results = calculate_results(y_true=val_labels, y_pred=baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

### Model 1: Simple dense model

In [30]:
# Create a tensorboard callback (need to create a new one for each model)
from helper_functions import create_tensorboard_callback

# create a directory to save TensorBoard logs
SAVE_DIR = "model_logs"

In [31]:
from tensorflow.keras import layers

inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)       # (None, 1) → (None, sequence_length)
x = embedding(x)                  # (None, sequence_length) → (None, sequence_length, embed_dim)
x = layers.GlobalAveragePooling1D()(x)  # (None, sequence_length, embed_dim) → (None, embed_dim)
outputs = layers.Dense(1, activation="sigmoid")(x)  # (None, embed_dim) → (None, 1)

model_1 = tf.keras.Model(inputs, outputs, name="model_1_dense")

In [32]:
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [33]:
# Compile the model
model_1.compile(
    loss="binary_crossentropy",  # NOT sparse_categorical or from_logits=True
    optimizer="adam",
    metrics=["accuracy"]
)

In [34]:
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [35]:
# Check intermediate shapes
print("After vectorizer:", text_vectorizer(tf.constant(["test"])).shape)
print("Model summary:")
model_1.summary()

After vectorizer: (1, 15)
Model summary:
Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params

In [36]:
# Fit the model
model_1_history = model_1.fit(x=train_sentences, # input sentences can be a list of strings due to text preprocessing layer built-in model
                              y=train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR, 
                                                                     experiment_name="simple_dense_model")])

Saving TensorBoard log files to: model_logs/simple_dense_model/20260327-141812
Epoch 1/5

215/215 [==============================] - 2s 5ms/step - loss: 0.6109 - accuracy: 0.6897 - val_loss: 0.5357 - val_accuracy: 0.7638
Epoch 2/5
215/215 [==============================] - 1s 5ms/step - loss: 0.4421 - accuracy: 0.8181 - val_loss: 0.4711 - val_accuracy: 0.7848
Epoch 3/5
215/215 [==============================] - 1s 5ms/step - loss: 0.3483 - accuracy: 0.8603 - val_loss: 0.4593 - val_accuracy: 0.7874
Epoch 4/5
215/215 [==============================] - 1s 4ms/step - loss: 0.2849 - accuracy: 0.8885 - val_loss: 0.4675 - val_accuracy: 0.7900
Epoch 5/5
215/215 [==============================] - 1s 4ms/step - loss: 0.2384 - accuracy: 0.9123 - val_loss: 0.4861 - val_accuracy: 0.7927


In [37]:
# Check the results
model_1.evaluate(val_sentences, val_labels)

24/24 [==============================] - 0s 1ms/step - loss: 0.4861 - accuracy: 0.7927


[0.48612740635871887, 0.7926509380340576]

In [38]:
embedding.weights

[<tf.Variable 'embedding/embeddings:0' shape=(10000, 128) dtype=float32, numpy=
 array([[-0.01907581,  0.00364587, -0.06849881, ..., -0.06739741,
          0.00274941, -0.01873689],
        [ 0.02591755,  0.04822931,  0.02994462, ..., -0.01823166,
          0.04686132, -0.03447028],
        [-0.06727766, -0.01382342, -0.0621584 , ..., -0.00312383,
          0.00410605,  0.04049176],
        ...,
        [-0.0489732 ,  0.0309372 ,  0.04782495, ...,  0.02571476,
          0.02978462, -0.03877405],
        [-0.00261174, -0.00785875, -0.05754837, ..., -0.01793123,
          0.08745345,  0.03435765],
        [-0.05119516,  0.10380791, -0.0806383 , ..., -0.05325021,
          0.08988879,  0.06160162]], dtype=float32)>]

In [39]:
# Make some predictions with our new model and evaluate those predictions
model_1_pred_probs = model_1.predict(val_sentences)
model_1_pred_probs.shape

24/24 [==============================] - 0s 982us/step


(762, 1)

In [40]:
model_1_pred_probs[0]

array([0.3327444], dtype=float32)

In [41]:
model_1_pred_probs[:10]

array([[0.3327444 ],
       [0.7745432 ],
       [0.99764   ],
       [0.08427956],
       [0.10257562],
       [0.93424004],
       [0.9077502 ],
       [0.99247575],
       [0.9605809 ],
       [0.21430244]], dtype=float32)

In [42]:
# Convert prediction probabilities to labels
model_1_preds = tf.round(model_1_pred_probs)  # rounds 0.5+ to 1, below 0.5 to 0
model_1_preds.shape  # (762, 1) — still fine

# Flatten if needed for evaluation
model_1_preds = tf.squeeze(model_1_preds)  # (762,) — matches val_labels shape
model_1_preds.shape  # (762,)

TensorShape([762])

In [43]:
from sklearn.metrics import classification_report

print(classification_report(val_labels, model_1_preds))

              precision    recall  f1-score   support

           0       0.76      0.90      0.82       414
           1       0.85      0.67      0.75       348

    accuracy                           0.79       762
   macro avg       0.80      0.78      0.79       762
weighted avg       0.80      0.79      0.79       762



In [44]:
# Convert model prediction probabilities to label format
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:20]

<tf.Tensor: shape=(20,), dtype=float32, numpy=
array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0.], dtype=float32)>

In [45]:
# Calculate our model_1 results
from pyexpat import model


model_1_results = calculate_results(y_true=val_labels, y_pred=model_1_preds)
print(model_1_results)

{'accuracy': 79.26509186351706, 'precision': 0.8008492102900203, 'recall': 0.7926509186351706, 'f1': 0.7888220986443327}


In [46]:
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}

In [47]:
import numpy as np
np.array(list(model_1_results.values()) > np.array(list(baseline_results.values())))


array([False, False, False,  True])

## Visualize learned embeddings

In [48]:
# get the vocabulary from the text vectorization
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:20]

(10000,
 ['',
  '[UNK]',
  'the',
  'a',
  'in',
  'to',
  'of',
  'and',
  'i',
  'is',
  'for',
  'on',
  'you',
  'my',
  'with',
  'it',
  'that',
  'at',
  'by',
  'this'])

In [51]:
# Model1 summary
model_1.summary()

Model: "model_1_dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [55]:
# Get the weight matrix of embedding layer
# (these are the numerical representations of each token in our training data, wich have been learned for 5 epochs)
embed_weights = model_1.get_layer('embedding').get_weights()[0]
print(embed_weights.shape) # same size as vocab size and embedding dim (output dim of our embedding layer)

(10000, 128)


Now we've got the embedding matrix our model has learned to represent our tokens, let's see how we can visualize it.

To do so, TensorFlow has a handy tool called projector: http://projector.tensorflow.org/

And TensorFlow also has an incredible guide on word embeddings themselves: https://www.tensorflow.org/text/tutorials/word_embeddings

In [ ]:
# Create embedding files (we got this from TensorFlo's word embedding documentation )

import io
out_v = io.open('vectors.tsv', 'w', encoding='utf-8') # write vectors.tsv
out_m = io.open('metadata.tsv', 'w', encoding='utf-8') # write metadata.tsv

for index, word in enumerate(words_in_vocab):
  if index == 0:
    continue  # skip 0, it's padding.
  vec = embed_weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + "\n")
  out_m.write(word + "\n")
out_v.close()
out_m.close()


## Recurrent Neural Networks (RNN's)

Recurrent Neural Networks (RNN's)

RNN's are useful for sequence data.

The premise of a recurrent neural network is to use the representation of a previous input to aid the representation of a later input.

If you want an overview of the internals of a recurrent neural network, see the following:
- MIT's sequence modelling lecture https://youtu.be/qjrad0V0uJE
- Chris Olah's intro to LSTMs: https://colah.github.io/posts/2015-08-Understanding-LSTMs/
- Andrej Karpathy's the unreasonable effectiveness of recurrent neural networks: http://karpathy.github.io/2015/05/21/rnn-effectiveness/

### Model 2: LSTM

LSTM = long short term memory (one of the most popular LSTM cells)

Our structure of an RNN typically looks like this:

```
Input (text) -> Tokenize -> Embedding -> Layers (RNNs/dense) -> Output (label probability)
```

In [60]:
# Create an LSTM model
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype='string')
x = text_vectorizer(inputs)
x = embedding(x)
# print(x.shape)
# x = layers.LSTM(64, return_sequences=True)(x) # when you're stacking RNN cells together, you will need to return_sequences=True 
# print(x.shape)
x = layers.LSTM(64)(x)
# print(x.shape)
# x = layers.Dense(64, activation='relu')(x)
# print(x.shape)
outputs = layers.Dense(1, activation='sigmoid')(x)
model_2 = tf.keras.Model(inputs, outputs, name='model_2_LSTM')

In [61]:
# get a summary of the model
model_2.summary()

Model: "model_2_LSTM"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_1 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 embedding (Embedding)       (None, 15, 128)           1280000   
                                                                 
 lstm_5 (LSTM)               (None, 64)                49408     
                                                                 
 dense_5 (Dense)             (None, 1)                 65        
                                                                 
Total params: 1329473 (5.07 MB)
Trainable params: 1329473 (5.07 MB)
Non-trainable params: 0 (0.00 Byte)
________________

In [71]:
# Compile the model
model_2.compile(loss='binary_crossentropy', 
                optimizer='adam', 
                metrics=['accuracy'])

In [72]:
# Fit the model
model_2_history = model_2.fit(train_sentences,
                                train_labels, 
                                epochs=5, 
                                validation_data=(val_sentences, val_labels),
                                callbacks=[create_tensorboard_callback(SAVE_DIR,
                                                                    'model_2_LSTM')])

Saving TensorBoard log files to: model_logs/model_2_LSTM/20260327-163615
Epoch 1/5
215/215 [==============================] - 3s 9ms/step - loss: 6.1665 - accuracy: 0.8476 - val_loss: 1.1473 - val_accuracy: 0.7730
Epoch 2/5
215/215 [==============================] - 2s 8ms/step - loss: 0.2446 - accuracy: 0.9183 - val_loss: 0.7109 - val_accuracy: 0.7861
Epoch 3/5
215/215 [==============================] - 2s 8ms/step - loss: 0.1506 - accuracy: 0.9425 - val_loss: 0.6923 - val_accuracy: 0.7651
Epoch 4/5
215/215 [==============================] - 2s 8ms/step - loss: 0.1227 - accuracy: 0.9521 - val_loss: 0.7618 - val_accuracy: 0.7848
Epoch 5/5
215/215 [==============================] - 2s 8ms/step - loss: 0.1019 - accuracy: 0.9597 - val_loss: 0.7660 - val_accuracy: 0.7874


In [73]:
# make predictions with LSTM model
model_2_pred_probs = model_2.predict(val_sentences)
model_2_pred_probs[:10]

24/24 [==============================] - 0s 2ms/step


array([[0.05392485],
       [0.885643  ],
       [0.99966466],
       [0.0409932 ],
       [0.00716625],
       [0.99934316],
       [0.9537101 ],
       [0.9999805 ],
       [0.99993557],
       [0.635383  ]], dtype=float32)

In [74]:
# Convert model 2 pred probs to labels
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))
model_2_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [75]:
# Calculate model 2 results
model_2_results = calculate_results(y_true=val_labels, 
                                    y_pred=model_2_preds)
model_2_results

{'accuracy': 78.74015748031496,
 'precision': 0.7877428758427124,
 'recall': 0.7874015748031497,
 'f1': 0.7863300776686603}

In [76]:
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1': 0.7862189758049549}